# Human 배치 모델 — L2 정규화 비교
## 1. 왜 / 무엇
보드를 보는 모델은 Train 배치를96% 맞췄지만 새 게임은13% 정도였습니다. Validation NLL은6epoch 이후 계속 증가했습니다.
이번에는 **L2 alpha만 바꿔, 과도한 학습을 억제하면 새 게임 예측도 좋아지는지** 확인합니다.

| 조건 | alpha | 나머지 |
|---|---:|---|
| 기존 재현 | 0.0001 | 모두 동일 |
| 정규화 강화 | 0.01 | 모두 동일 |
| 더 강한 정규화 | 1.0 | 모두 동일 |

모든 조건을30epoch까지 학습합니다. 조기 종료/모델 크기/학습률을 함께 바꾸지 않습니다.
별도 Smoke 학습은 없고, 기본 모두 실행은 내부 검사만 합니다.
[명세](../../../ai-harness/ml-experiments/experiments/human-placement-l2-regularization.md)


## 2. 손으로 이해하기
같은 배치 확률을 내는 두 모델이라면, L2는 가중치가 과도하게 커진 모델에 더 큰 벌점을 줍니다.
예를 들어 가중치 제곱합이4라면 alpha0.01보다1.0에서 L2 항이100배 강합니다.
실제 최적화의 정규화 항은 표본 weight 합으로 나뉘므로 alpha 자체를 loss로 해석하지 않습니다.

- 비교할 예측 손실은 **L2를 제외한 같은 정의의 NLL**입니다. 최적화 loss는 정규화 항이 달라 직접 성능 비교에 쓰지 않습니다.
- NLL은 정답에 준 확률, Top-1은1등 답 일치 여부입니다. NLL만 좋아지고 Top-1이 나빠지면 trade-off입니다.
- Validation NLL은 known1364구간, 정확도는 OOV152개를 포함한1516구간 전체입니다.
- 출력은 Train에서 본2133배치 class이지 합법/도달 가능한 후보 목록이 아닙니다.


In [ ]:
import contextlib
import copy
import hashlib
import io
import json
import os
import platform
import warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.neural_network import MLPClassifier

ROOT = next(p.resolve() for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "ai-harness").is_dir() and (p / "package.json").is_file())
EXPERIMENT_ID = "human-placement-l2-regularization"
SOURCE_NOTEBOOK = ROOT / "scripts/ml/notebooks/human_placement_l2_regularization.ipynb"
SNAPSHOT_NOTEBOOK = ROOT / "scripts/ml/notebooks/human_placement_snapshot_baseline.ipynb"
SNAPSHOT_DEFS_SHA = "233ceca8322bd4335b798cdd452064d389e0ddbce6451610c7ba7d56a1dd9c5c"
REFERENCE_PATH = ROOT / "data/ml-experiments/human-placement-snapshot-baseline/full-report-run2.json"
REFERENCE_SHA = "d87456bbb3d4f289db38d1bc70d8b6b8d1615f34a5bd4ddaa3f025d40f6fb19e"
selected = [{"id": c["id"], "source": "".join(c["source"])}
            for c in json.loads(SNAPSHOT_NOTEBOOK.read_text(encoding="utf-8"))["cells"]
            if c["cell_type"] == "code" and "definitions" in c.get("metadata", {}).get("tags", [])]
assert hashlib.sha256((json.dumps(selected, ensure_ascii=False, sort_keys=True,
                                  allow_nan=False, indent=2) + "\n").encode()).hexdigest() == SNAPSHOT_DEFS_SHA
shared = {"__name__": "l2_snapshot_shared"}
for c in selected:
    exec(compile(c["source"], str(SNAPSHOT_NOTEBOOK), "exec"), shared)
shared["EXPERIMENT_ID"], shared["SOURCE_NOTEBOOK"] = EXPERIMENT_ID, SOURCE_NOTEBOOK
shared["base"]["dependency"]["link"]["dataset"]["shared"]["EXPERIMENT_ID"] = EXPERIMENT_ID
for name in ("require", "json_bytes", "sha256_file", "definition_sha", "threadpool_limits",
             "rank_probabilities", "metric_summary", "evaluate_rankings", "predict_probabilities",
             "report_path", "publish_report"):
    globals()[name] = shared[name]
ALPHAS = (0.0001, 0.01, 1.0)
EPOCHS = 30


In [ ]:
def make_model(alpha):
    require(type(alpha) in (int, float) and alpha in ALPHAS, "ALPHA_OUTSIDE_FIXED_COMPARISON")
    return MLPClassifier(hidden_layer_sizes=(64,), activation="relu", solver="adam",
        alpha=alpha, batch_size=128, learning_rate_init=0.001, shuffle=True,
        random_state=1729, early_stopping=False)

def fit_candidate(train_x, y, validation_x, train_rows, validation_rows, vocabulary, alpha, epochs):
    """Train만으로 업데이트하고 같은 epoch 종료 상태에서 두 split을 평가한다."""
    require(type(epochs) is int and epochs > 0, "INVALID_EPOCH_BUDGET")
    model = make_model(alpha)
    history, losses = [], []
    for epoch in range(epochs):
        with warnings.catch_warnings(record=True) as caught:
            warnings.simplefilter("always")
            model.partial_fit(train_x, y, classes=np.arange(len(vocabulary)) if epoch == 0 else None)
        require(not caught, "UNEXPECTED_FIT_WARNING: " + "; ".join(
            f"{w.category.__name__}: {w.message}" for w in caught))
        require(np.isfinite(model.loss_), "NONFINITE_TRAIN_LOSS")
        losses.append(float(model.loss_))
        train_ranked = rank_probabilities(train_rows,
            predict_probabilities(model, train_x, len(vocabulary)), vocabulary)
        validation_ranked = rank_probabilities(validation_rows,
            predict_probabilities(model, validation_x, len(vocabulary)), vocabulary)
        history.append({"epoch": epoch + 1, "train": metric_summary(train_ranked),
                        "validation": metric_summary(validation_ranked)})
        print(f"alpha={alpha:g} epoch={epoch + 1}/{epochs}: "
              f"train NLL={history[-1]['train']['knownTargetNll']}, "
              f"val NLL={history[-1]['validation']['knownTargetNll']}")
    require(all(np.isfinite(a).all() for a in (*model.coefs_, *model.intercepts_)), "NONFINITE_MODEL")
    result = evaluate_rankings(validation_ranked)
    require(history[-1]["validation"] == result["segmentMicro"], "FINAL_EPOCH_METRIC_MISMATCH")
    return {"result": result, "history": history, "trainingLoss": losses,
            "coefficientL2": float(np.sqrt(sum(np.square(a).sum() for a in model.coefs_)))}

def close_numeric(actual, expected):
    """구조/정수는 일치시키고 실수는 절대오차1e-12 이내인지 검사한다."""
    if isinstance(expected, dict):
        return (isinstance(actual, dict) and actual.keys() == expected.keys()
                and all(close_numeric(actual[k], v) for k, v in expected.items()))
    if isinstance(expected, list):
        return (isinstance(actual, list) and len(actual) == len(expected)
                and all(close_numeric(a, b) for a, b in zip(actual, expected)))
    if type(expected) is float:
        return type(actual) in (int, float) and np.isfinite(actual) and abs(actual - expected) <= 1e-12
    return type(actual) is type(expected) and actual == expected

def load_reference():
    require(sha256_file(REFERENCE_PATH) == REFERENCE_SHA, "REFERENCE_SHA_MISMATCH")
    ref = json.loads(REFERENCE_PATH.read_text(encoding="utf-8"), parse_constant=shared["reject_constant"])
    require(ref["reportSchemaVersion"] == 2 and ref["runMode"] == "full"
            and ref["fullCohortCovered"] is True and ref["sourceDefinitionSha256"] == SNAPSHOT_DEFS_SHA
            and ref["baseDefinitionSha256"] == shared["BASE_DEFS_SHA"], "REFERENCE_CONTRACT_MISMATCH")
    return ref

def compare_results(results):
    baseline = results["alpha_0.0001"]["result"]
    rows = []
    for alpha in ALPHAS:
        key = f"alpha_{alpha:g}"
        candidate = results[key]["result"]
        rows.append({"alpha": alpha, **{
            "delta_" + m: candidate["gameMacro"][m] - baseline["gameMacro"][m]
            for m in ("top1", "top3", "mrr")},
            "delta_knownNll": candidate["segmentMicro"]["knownTargetNll"]
                              - baseline["segmentMicro"]["knownTargetNll"]})
    return rows


In [ ]:
def run_full():
    require(ALPHAS == (0.0001, 0.01, 1.0) and EPOCHS == 30, "EXPERIMENT_SETTINGS_CHANGED")
    require((platform.python_version(), np.__version__, shared["base"]["sklearn"].__version__)
            == ("3.12.13", "2.5.1", "1.9.0"), "PINNED_ENVIRONMENT_REQUIRED")
    source_sha = definition_sha(SOURCE_NOTEBOOK)
    require(definition_sha(SNAPSHOT_NOTEBOOK) == SNAPSHOT_DEFS_SHA, "SNAPSHOT_DEFINITIONS_CHANGED")
    reference = load_reference()
    data, watched = shared["load_snapshots"]()
    vocabulary = shared["train_vocabulary"](data["train"])
    require(len(vocabulary) == 2133, "TRAIN_VOCABULARY_CHANGED")
    ids = {key: i for i, key in enumerate(vocabulary)}
    y = np.array([ids[r["targetKey"]] for r in data["train"]], dtype=int)
    train_x = np.stack([r["x"] for r in data["train"]])
    scaler = shared["base"]["StandardScaler"]().fit(train_x)
    train_x = shared["masked_features"](scaler.transform(train_x), "board_piece")
    validation_x = shared["masked_features"](
        scaler.transform(np.stack([r["x"] for r in data["validation"]])), "board_piece")
    require(np.allclose(scaler.mean_, reference["training"]["scalerMean"], rtol=0, atol=1e-12)
            and np.allclose(scaler.scale_, reference["training"]["scalerScale"], rtol=0, atol=1e-12)
            and hashlib.sha256(json_bytes(vocabulary)).hexdigest() == reference["vocabulary"]["sha256"],
            "PREPROCESSING_DRIFT")
    results = {}
    with threadpool_limits(limits=1):
        for alpha in ALPHAS:
            key = f"alpha_{alpha:g}"
            results[key] = fit_candidate(train_x, y, validation_x, data["train"],
                                         data["validation"], vocabulary, alpha, EPOCHS)
            require(all(h["train"]["knownNllDenominator"] == 6417
                        and h["validation"]["segments"] == 1516
                        and h["validation"]["knownNllDenominator"] == 1364
                        and h["validation"]["unknownTargetSegments"] == 152
                        for h in results[key]["history"]), "METRIC_DENOMINATOR_CHANGED")
            if alpha == ALPHAS[0]:
                require(close_numeric(results[key]["result"], reference["results"]["board_piece"])
                        and close_numeric(results[key]["history"], reference["epochMetrics"]["board_piece"])
                        and close_numeric(results[key]["trainingLoss"], reference["trainingLoss"]["board_piece"]),
                        "BASELINE_REPRODUCTION_FAILED")
                print("Baseline reproduced within 1e-12. Continuing fixed alpha comparison.")
    per_game_delta = []
    for alpha in ALPHAS[1:]:
        key = f"alpha_{alpha:g}"
        for a, b in zip(results["alpha_0.0001"]["result"]["perGame"], results[key]["result"]["perGame"]):
            require(a["gameId"] == b["gameId"], "GAME_ORDER_CHANGED")
            per_game_delta.append({"alpha": alpha, "gameId": a["gameId"], "segments": a["segments"],
                **{"delta_" + m: b[m] - a[m] for m in ("top1", "top3", "mrr")}})
    require(all(sha256_file(p) == h for p, h in watched.items())
            and sha256_file(REFERENCE_PATH) == REFERENCE_SHA
            and definition_sha(SOURCE_NOTEBOOK) == source_sha
            and definition_sha(SNAPSHOT_NOTEBOOK) == SNAPSHOT_DEFS_SHA
            and definition_sha(shared["BASE_NOTEBOOK"]) == shared["BASE_DEFS_SHA"], "INPUT_OR_CODE_CHANGED")
    return {"reportSchemaVersion": 1, "experimentId": EXPERIMENT_ID, "runMode": "full",
        "executionPolicy": "full-only", "fullCohortCovered": True, "policyDecisionEligible": True,
        "canonicalResultEligible": False, "productionIntegration": False, "inferenceEligible": False,
        "wholePolicyEvidence": False, "legalityChecked": False, "reachableCandidatesGenerated": False,
        "snapshotObservation": "unverified", "analysisStatus": "l2_comparison_requires_review",
        "modelFitCount": 3, "smokeFitCount": 0, "alphas": list(ALPHAS), "results": results,
        "comparisons": compare_results(results), "perGameDelta": per_game_delta,
        "baselineReproduced": True, "baselineTolerance": 1e-12, "bestEpochSelected": False,
        "validationRole": "reused_development_comparison_not_independent_test",
        "training": {**{k: v for k, v in reference["training"].items() if k != "alpha"},
                     "scalerMean": scaler.mean_.tolist(), "scalerScale": scaler.scale_.tolist()},
        "vocabulary": reference["vocabulary"], "provenance": reference["provenance"],
        "epochMetricContract": reference["epochMetricContract"], "environment": reference["environment"],
        "sourceDefinitionSha256": source_sha, "snapshotDefinitionsSha256": SNAPSHOT_DEFS_SHA,
        "referenceSha256": REFERENCE_SHA, "sourceUnchanged": True,
        "testFileRead": False, "rawFileRead": False, "newConditionRepeatDeterminismChecked": False}


In [ ]:
def internal_checks():
    """실제 학습 없이 공유 계약과 alpha 외 동일 설정/업데이트 경계를 검사한다."""
    with contextlib.redirect_stdout(io.StringIO()):
        shared["internal_checks"]()
    parameters = [make_model(a).get_params() for a in ALPHAS]
    for p, alpha in zip(parameters, ALPHAS):
        require(p["alpha"] == alpha, "MODEL_ALPHA_MISMATCH")
    assert all({k: v for k, v in p.items() if k != "alpha"} ==
               {k: v for k, v in parameters[0].items() if k != "alpha"} for p in parameters)
    for invalid in (True, -1, float("nan"), 0.1):
        try:
            make_model(invalid)
        except ValueError:
            pass
        else:
            raise AssertionError("UNDECLARED_ALPHA_ALLOWED")
    assert close_numeric({"x": [1.0, None, 3]}, {"x": [1.0 + 5e-13, None, 3]})
    assert not close_numeric([1.0], [1.0 + 2e-12])
    assert not close_numeric([float("nan")], [1.0])
    assert not close_numeric({"x": 1}, {"x": 1, "y": 2})
    train_x, labels = np.ones((2, 207)), np.array([0, 1])
    vocabulary = ["square|a", "square|b"]
    train_rows = [{"gameId": "Train", "pieceType": "square", "targetKey": k} for k in vocabulary]
    val_rows = [{"gameId": "Val", "pieceType": "square", "targetKey": k}
                for k in (vocabulary[0], "square|OOV")]
    models = []
    failure = None
    class FakeModel:
        classes_ = np.arange(2)
        coefs_, intercepts_ = [np.ones((1, 1))], [np.zeros(1)]
        def __init__(self, alpha):
            self.alpha, self.steps = alpha, 0
            models.append(self)
        def partial_fit(self, x, y, classes=None):
            assert np.array_equal(x, train_x) and np.array_equal(y, labels)
            assert (classes is None) == (self.steps > 0)
            self.steps += 1
            self.loss_ = np.float64(1 / self.steps)
            if self.steps == 2 and failure == "nonfinite":
                self.loss_ = np.float64(np.inf)
            if self.steps == 2 and failure == "warning":
                warnings.warn("internal-check", RuntimeWarning)
            return self
        def predict_proba(self, x):
            return np.tile([0.75, 0.25], (len(x), 1))
    original = globals()["make_model"]
    try:
        globals()["make_model"] = FakeModel
        with contextlib.redirect_stdout(io.StringIO()):
            result = fit_candidate(train_x, labels, np.zeros((2, 207)), train_rows,
                                   val_rows, vocabulary, 0.01, 2)
        assert models[-1].steps == 2 and models[-1].alpha == 0.01
        assert [h["epoch"] for h in result["history"]] == [1, 2]
        for h in result["history"]:
            assert np.isclose(h["train"]["knownTargetNll"], -(np.log(.75) + np.log(.25)) / 2)
            assert np.isclose(h["validation"]["knownTargetNll"], -np.log(.75))
            assert h["validation"]["knownNllDenominator"] == 1 and h["validation"]["unknownTargetSegments"] == 1
            assert h["validation"]["top1"] == .5 and h["validation"]["fullTargetNll"] is None
        assert result["trainingLoss"] == [1.0, .5]
        assert result["result"]["segmentMicro"] == result["history"][-1]["validation"]
        for failure, reason in (("nonfinite", "NONFINITE_TRAIN_LOSS"),
                                ("warning", "UNEXPECTED_FIT_WARNING: RuntimeWarning: internal-check")):
            try:
                with contextlib.redirect_stdout(io.StringIO()):
                    fit_candidate(train_x, labels, np.zeros((2, 207)), train_rows,
                                  val_rows, vocabulary, 0.01, 2)
            except ValueError as error:
                assert str(error) == reason
            else:
                raise AssertionError("INVALID_TRAIN_STEP_ACCEPTED")
    finally:
        globals()["make_model"] = original
    return "PASS: shared input/ranking/path checks; alpha-only parameters; NumPy loss; Train-only updates; NLL/OOV; warnings; reproduction tolerance. No real fit."


In [ ]:
def show_report(report):
    require(report["experimentId"] == EXPERIMENT_ID and report["baselineReproduced"], "REPORT_CONTRACT_INVALID")
    summary, diagnostics = [], []
    fig, axes = plt.subplots(2, 3, figsize=(16, 8))
    for col, alpha in enumerate(ALPHAS):
        key = f"alpha_{alpha:g}"
        item = report["results"][key]
        result, history = item["result"], item["history"]
        summary.append({"alpha": alpha, **{"valMacro_" + m: result["gameMacro"][m]
                        for m in ("top1", "top3", "mrr")}, "valKnownNll": result["segmentMicro"]["knownTargetNll"]})
        minimum = min(history, key=lambda h: (h["validation"]["knownTargetNll"], h["epoch"]))
        diagnostics.append({"alpha": alpha, "trainTop1": history[-1]["train"]["top1"],
            "valTop1": history[-1]["validation"]["top1"],
            "trainNll": history[-1]["train"]["knownTargetNll"],
            "valKnownNll": history[-1]["validation"]["knownTargetNll"],
            "minValNllEpoch_diagnosticOnly": minimum["epoch"], "coefficientL2": item["coefficientL2"]})
        for split, style in (("train", "-"), ("validation", "--")):
            last = history[-1][split]
            x = [h["epoch"] for h in history]
            axes[0, col].plot(x, [h[split]["knownTargetNll"] for h in history], style,
                             label=f"{split} known n={last['knownNllDenominator']}/{last['segments']}")
            axes[1, col].plot(x, [h[split]["top1"] for h in history], style, label=split)
        axes[0, col].set(title=f"alpha={alpha:g}", xlabel="epoch", ylabel="Known-target NLL, no L2")
        axes[1, col].set(xlabel="epoch", ylabel="Top-1, all targets", ylim=(0, 1))
        axes[0, col].legend(fontsize=8); axes[1, col].legend()
    print(report["analysisStatus"], "| same Validation reused for development; NOT gameplay evidence")
    display(pd.DataFrame(summary).set_index("alpha"))
    display(pd.DataFrame(report["comparisons"]).set_index("alpha"))
    display(pd.DataFrame(diagnostics).set_index("alpha"))
    print("Val NLL: known1364; Val accuracy: all1516 incl.152 OOV. Minimum epoch is not a selected checkpoint.")
    fig.tight_layout(); plt.show(); plt.close(fig)
    deltas = pd.DataFrame(report["perGameDelta"])
    display(deltas)
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    for ax, alpha in zip(axes, ALPHAS[1:]):
        rows = deltas[deltas["alpha"] == alpha]
        ax.bar(range(len(rows)), rows["delta_top1"])
        ax.axhline(0, color="black", linewidth=.8)
        ax.set(title=f"alpha={alpha:g} minus baseline", xlabel="Game index (Unicode order)",
               ylabel="Top-1 delta; positive is higher")
    fig.tight_layout(); plt.show(); plt.close(fig)


## 3. 내부 검사 / Full 실행
기본은 내부 assert만 실행합니다. 작은 실제 모델 학습은 없습니다.
Full은3조건×30epoch입니다. 기준선 재현에 실패하면 후보 조건으로 진행하지 않습니다.

아래에서 RUN_FULL=True로 바꾸고 새 FULL_REPORT 경로를 지정한 뒤 모두 실행하세요.
기존 파일이 있으면 run2 같은 새 이름을 사용합니다. 기존 source Notebook/보고서는 수정하지 않습니다.


In [ ]:
RUN_FULL = False
FULL_REPORT = os.environ.get("HUMAN_PLACEMENT_L2_REPORT",
    "data/ml-experiments/human-placement-l2-regularization/full-report-run1.json")


In [ ]:
print(internal_checks())
full_report = None
if RUN_FULL:
    destination = report_path(FULL_REPORT)
    full_report = run_full()
    publish_report(full_report, destination)
    print("Saved:", str(destination), "| SHA-256:", sha256_file(destination))
else:
    print("Full disabled. No dataset/reference report access or neural training.")


## 4. 결과 읽기
이미 실행했다면 RUN_FULL=False를 유지하고 아래 REPORT_TO_READ에 JSON 경로를 넣으세요.
첫 표는30epoch의 게임 평균 정확도/순위, 다음 표는 기준선 대비 차이입니다.
곡선은 구간 기준 Train/Val NLL과 정확도입니다. 더 작은 gap만으로 성공이라 하지 않습니다.
모두 못 맞추게 돼도 gap은 작아질 수 있으므로 Validation 정확도/MRR와 NLL을 함께 봅니다.


In [ ]:
REPORT_TO_READ = None
if REPORT_TO_READ:
    path = Path(REPORT_TO_READ)
    if not path.is_absolute():
        path = ROOT / path
    loaded = json.loads(path.read_text(encoding="utf-8"), parse_constant=shared["reject_constant"])
    require(loaded["reportSchemaVersion"] == 1 and loaded["experimentId"] == EXPERIMENT_ID
            and loaded["runMode"] == "full" and loaded["alphas"] == list(ALPHAS)
            and loaded["sourceDefinitionSha256"] == definition_sha(SOURCE_NOTEBOOK)
            and loaded["snapshotDefinitionsSha256"] == SNAPSHOT_DEFS_SHA
            and loaded["referenceSha256"] == REFERENCE_SHA, "STALE_OR_DIFFERENT_REPORT")
    full_report = loaded


In [ ]:
if full_report is not None:
    show_report(full_report)


## 5. 한계 / 다음 gate
- 같은 Validation을 반복해서 본 뒤 정한 개발 실험입니다. 독립 Test 성능/통계적 우월/개인 복제를 주장하지 않습니다.
- 정규화를 강화하면 과적합이 줄 수도 있지만 구분력까지 떨어질 수 있습니다. 낮은 NLL만으로 후보를 자동 채택하지 않습니다.
- 단일 seed·단일 Full이고 새 조건의 반복 결정성/동결 검증은 아닙니다. 최저 NLL epoch를 골라 결과를 바꾸지 않습니다.
- OOV/희소2133-class·관측 시점·합법성/도달 가능성 한계는 그대로입니다. 새 로그 수집을 요구하거나 휴리스틱으로 정답을 바꾸지 않습니다.
- 출력은 report뿐입니다. 가중치/artifact/배치 controller/사이트 기능은 추가하지 않습니다.
- 다음은 사용자 Full의 정확도·MRR·NLL trade-off 해석이며 추가 alpha·재학습·runtime·commit을 자동 시작하지 않습니다.
